# Pretrain a small decoder-only LM from scratch

Run in Kaggle or Colab. This notebook uses the repository's original PyTorch decoder model and UTF-8 byte tokenizer, not the upstream project's source. It trains a small educational model with next-byte prediction. It is not a recipe for billion-parameter training. Select a GPU runtime for faster experiments; start with small settings and increase only if memory permits.

The default below creates a tiny demonstration corpus so the notebook can be smoke-tested. Replace `TEXT_FILE` with a UTF-8 plain-text dataset you have rights to use. The notebook writes checkpoints under the runtime's working directory; copy/download them to persistent storage before the session expires.

In [ ]:
import os, sys, json, platform, subprocess
from pathlib import Path
import torch

# Repository checkout. Change REPO_URL if you are using your own fork.
REPO_URL = "https://github.com/MKarthik730/large-language-models.git"
REPO_DIR = Path("/kaggle/working/large-language-models" if Path("/kaggle/working").exists() else "/content/large-language-models")
if not (REPO_DIR / "training" / "scripts" / "train_pretrain.py").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Python:", platform.python_version(), "PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))

# PyTorch is normally preinstalled in Kaggle/Colab runtimes. This notebook uses that runtime build.

In [ ]:
# Set this to your uploaded/mounted UTF-8 text file. Empty means use a toy demo corpus.
TEXT_FILE = ""
WORK_DIR = Path("/kaggle/working/training-run" if Path("/kaggle/working").exists() else "/content/training-run")
WORK_DIR.mkdir(parents=True, exist_ok=True)
if TEXT_FILE:
    text_path = Path(TEXT_FILE)
    if not text_path.is_file(): raise FileNotFoundError(text_path)
else:
    text_path = WORK_DIR / "demo.txt"
    text_path.write_text(("A transformer predicts the next byte from context. " * 300), encoding="utf-8")
print("Training text:", text_path, "bytes:", text_path.stat().st_size)

# Small defaults are suitable for a quick notebook demonstration.
MODEL_CONFIG = {
    "vocab_size": 260,
    "max_seq_len": 64,
    "d_model": 64,
    "n_heads": 4,
    "n_layers": 2,
    "dropout": 0.1,
}
TRAIN_CONFIG = {
    "batch_size": 8,
    "learning_rate": 0.0003,
    "steps": 100,
    "eval_interval": 20,
    "eval_batches": 3,
    "grad_clip": 1.0,
    "seed": 1337,
}
CONFIG_FILE = WORK_DIR / "config.json"
CONFIG_FILE.write_text(json.dumps({"model": MODEL_CONFIG, "training": TRAIN_CONFIG}, indent=2), encoding="utf-8")
CHECKPOINT = WORK_DIR / "base.pt"

In [ ]:
subprocess.run([
    sys.executable, "-m", "training.scripts.train_pretrain",
    "--data", str(text_path),
    "--output", str(CHECKPOINT),
    "--config", str(CONFIG_FILE),
    "--device", "cuda" if torch.cuda.is_available() else "cpu",
], cwd=REPO_DIR, check=True)
print("Base checkpoint:", CHECKPOINT, "size:", CHECKPOINT.stat().st_size)

In [ ]:
# Generate a short sample from the trained base checkpoint.
subprocess.run([
    sys.executable, "-m", "training.scripts.generate",
    "--checkpoint", str(CHECKPOINT),
    "--prompt", "A transformer",
    "--max-new-tokens", "80",
    "--device", "cuda" if torch.cuda.is_available() else "cpu",
], cwd=REPO_DIR, check=True)

## Save the checkpoint

Kaggle: create a notebook version with outputs or copy the checkpoint to a Kaggle Dataset. Colab: download the checkpoint or copy it to mounted Google Drive. Runtime-local files are lost when the session ends. Use properly licensed text, and do not treat training loss or a short demo as a meaningful quality/safety evaluation.

In [ ]:
# Optional Colab download.
try:
    from google.colab import files
except ImportError:
    print("Download manually from", CHECKPOINT)
else:
    files.download(str(CHECKPOINT))